# Llama 3.1 Unified Prompt Architecture
## Consolidated Multi-Agent Emotion Recognition System

This notebook condenses the multi-agent LLM architecture into a single unified Llama 3.1 prompt that integrates:
- **Character Profiling**: Behavioral analysis against character baseline
- **Context Management**: Emotional scene vibe and trajectory tracking
- **Emotional Shift Detection**: Temporal dynamics and pivot detection
- **Linguistic Pragmatics**: Fine-grained lexical and syntactic analysis
- **Relational Dynamics**: Interpersonal context and vulnerability detection
- **Social Dynamics**: Social intent and face management analysis
- **Council Aggregation**: Synthesized emotion classification with CoT reasoning

All steps are embedded in a single optimized prompt for Llama 3.1, making predictions faster and more efficient.

In [1]:
import os
import json
import re
from datetime import datetime
from pathlib import Path
from typing import Optional, Dict, Any

import pandas as pd
import sys
sys.path.append("../")

import vertexai
from dotenv import load_dotenv
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from tqdm import tqdm

load_dotenv()

# Configuration
BASE_DIR = "../"
DATA_PATH = os.path.join(BASE_DIR, "data", "test_sent_emo.csv")
BIO_CARDS_PATH = os.path.join(BASE_DIR, "logs", "speaker_bio_cards.json")
OUTPUT_DIR = os.path.join(BASE_DIR, "logs", "llama31_unified")
PROMPTS_DIR = os.path.join(BASE_DIR, "src", "llama3_prompts")

# Vertex AI Configuration
PROJECT_ID = os.getenv("LLAMA_MODEL_PROJECT_ID") or os.getenv("TUNED_MODEL_PROJECT_ID")
LOCATION = os.getenv("VERTEX_LOCATION", "us-central1")
ENDPOINT_ID = os.getenv("LLAMA31_ENDPOINT_ID", "2346569469662330880")

if not PROJECT_ID:
    raise ValueError("Missing project id. Set LLAMA_MODEL_PROJECT_ID or TUNED_MODEL_PROJECT_ID in .env")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Initialize Vertex AI for Llama 3.1
vertexai.init(project=PROJECT_ID, location=LOCATION)
from vertexai.generative_models import GenerativeModel

llama31_model = GenerativeModel(f"projects/{PROJECT_ID}/locations/{LOCATION}/endpoints/{ENDPOINT_ID}")

print(f"✅ Configuration Complete")
print(f"  Project: {PROJECT_ID}")
print(f"  Location: {LOCATION}")
print(f"  Endpoint: {ENDPOINT_ID}")
print(f"  Data: {DATA_PATH}")
print(f"  Output: {OUTPUT_DIR}")

✅ Configuration Complete
  Project: project-77549f95-0391-4b29-911
  Location: us-central1
  Endpoint: 2346569469662330880
  Data: ../data\test_sent_emo.csv
  Output: ../logs\llama31_unified


## Section 2: Load Data and Speaker Context

Load emotion recognition dataset, speaker bio cards, and prepare enriched context for the unified prompt.

In [2]:
def load_data_from_csv(path: Path) -> pd.DataFrame:
    """Load and normalize emotion recognition data."""
    df = pd.read_csv(path)
    df = df.sort_values(["Dialogue_ID", "Utterance_ID"]).reset_index(drop=True)
    return df

def load_speaker_bio_cards(json_file: str = BIO_CARDS_PATH) -> Dict[str, Any]:
    """Load speaker biographical information."""
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"⚠️  Bio cards load error: {e}")
        return {}

def format_bio_cards_context(bio_cards: Dict, speakers: list) -> str:
    """Format speaker bio cards for the unified prompt."""
    if not bio_cards or not speakers:
        return ""
    
    context = "\n### SPEAKER PROFILES\n"
    for speaker in speakers:
        if speaker in bio_cards:
            bio = bio_cards[speaker]
            if isinstance(bio, dict):
                bio_text = "\n".join([f"{k}: {v}" for k, v in bio.items()])
            else:
                bio_text = str(bio)
            context += f"\n**{speaker}:**\n{bio_text}\n"
    return context

print("Loading dataset...")
df = load_data_from_csv(Path(DATA_PATH))

print("Loading speaker bio cards...")
bio_cards = load_speaker_bio_cards()

print(f"\n✅ Data Loaded")
print(f"  Total rows: {len(df)}")
print(f"  Unique dialogues: {df['Dialogue_ID'].nunique()}")
print(f"  Unique speakers: {len(bio_cards)}")
print(f"  Emotion distribution:\n{df['Emotion'].value_counts()}")

Loading dataset...
Loading speaker bio cards...

✅ Data Loaded
  Total rows: 2610
  Unique dialogues: 280
  Unique speakers: 260
  Emotion distribution:
Emotion
neutral     1256
joy          402
anger        345
surprise     281
sadness      208
disgust       68
fear          50
Name: count, dtype: int64


## Section 3: Build Unified Llama 3.1 System Prompt

Create a comprehensive system prompt that consolidates all 7 agent roles (Character Profiler, Context Manager, Emotional Shift Detector, Empathy Reasoner, Relational Graph, Social Dynamics Expert, Council Aggregator) into a single optimized instruction set for Llama 3.1.

In [3]:
# Unified Llama 3.1 System Prompt - Consolidates all 7 agent roles

UNIFIED_SYSTEM_PROMPT = """
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an EXPERT EMOTION RECOGNITION SYSTEM specialized in the MELD benchmark. 
Your task is to predict ONE emotion for EVERY utterance using ONLY these labels:
[anger, disgust, fear, joy, neutral, sadness, surprise]

You will analyze each utterance through 7 integrated analytical lenses in sequence, then synthesize into a final decision:

---

## AGENT 1: CHARACTER BEHAVIORAL SPECIALIST
Role: Analyze utterance against character's psychological baseline.

ANALYSIS:
1. Identify Character-Specific Triggers in the utterance
2. Compare actual utterance to expected baseline behavior
3. Check Intensity vs Baseline and Valence
4. Detect if sarcasm/humor masks Sadness, Fear, or Disgust
5. Rate Sarcasm Probability (0-1)

Output tags: [AGGRESSIVE/DEFENSIVE], [VULNERABLE/EXPOSED], [PLAYFUL/AFFECTIONATE], [BASELINE/NEUTRAL]

---

## AGENT 2: SCENE ARCHITECT & EMOTIONAL HISTORIAN
Role: Analyze conversation history and establish emotional context.

CONTEXT ANALYSIS:
1. Scene Vibe: 1-sentence summary (e.g., "Playful ribbing", "Tense confrontation")
2. Emotional Arc: Track baseline, inflection points, and momentum
3. Shift Warning (Yes/No): Has atmosphere abruptly changed?

Character Mood Assessment:
- mood_trend: escalating/de-escalating/stable/neutral-baseline
- arousal_level: low/medium/high
- valence: positive/negative/neutral

⚠️ CRITICAL: High Arousal + Negative Valence ≠ Anger (could be Panic/Despair)

Output tags: [SCENE: NEUTRAL] or describe vibe

---

## AGENT 3: TEMPORAL DYNAMICS & SHIFT DETECTOR
Role: Identify emotional continuity breaks vs. patterns.

SHIFT ANALYSIS:
1. Micro-Baseline: State of conversation one turn ago
2. Delta Detection:
   - Energy (Arousal) spike/drop?
   - Valence flip (Positive <-> Negative)?
   - Character-Pattern check: Exceeds baseline/habitual conflict style?
3. Trigger ID: Internal (memory), External (word/action), or Social (backpedaling)

⚠️ GATES:
- Anger shift only valid if DEVIATES from habitual Conflict Style
- Sincerity shift ≠ Fear. Fear requires anticipatory worry or reactive defense

Output tags: [SHIFT: FALSE - CONTINUITY MAINTAINED] OR [SHIFT: TRUE - EMOTIONAL PIVOT DETECTED]

---

## AGENT 4: LINGUISTIC PRAGMATICS & SUBTEXT SPECIALIST
Role: Extract emotional signals from linguistic features.

UTTERANCE DISSECTION:
1. Lexical: High-emotion keywords, negations, intensifiers, hedging
2. Syntactic: Sentence type, discourse markers, pragmatic disconnect
3. Paralinguistic: Punctuation (!!!, ???, ...), repetition, hesitation
4. Arousal assessment (1-10)
5. Surface Sentiment: Pos/Neg/Neu
6. Latent Sentiment: Pos/Neg/Neu
7. Irony/Sarcasm detection

Output tags: [LINGUISTIC TONE: AGGRESSIVE], [LINGUISTIC TONE: WITHDRAWN/VULNERABLE], [LINGUISTIC TONE: NEUTRAL]

---

## AGENT 5: RELATIONSHIP HISTORIAN & MEMORY SPECIALIST
Role: Map interpersonal dynamics using speaker relationship history.

RELATIONSHIP MAPPING:
1. Bond & History: Primary type and pivotal moments
2. Interaction Patterns:
   - Default mask with this listener
   - Psychological Safety (1-10): Likelihood of expressing vulnerability
   - Vulnerability Capacity: Masking Sadness/Fear with Anger/Sarcasm?
3. Relational Influence (1-10): How much listener changes speaker's persona

Output tags: [RELATIONSHIP DYNAMIC: DEFENSIVE] OR [RELATIONSHIP DYNAMIC: SAFE/SUPPORTIVE]

---

## AGENT 6: INTERPERSONAL STRATEGIST & FACE-WORK ANALYST
Role: Analyze social moves, face management, and conflict vs. vulnerability.

SOCIAL FRAMEWORK:
1. Power: Hierarchical position and power moves
2. Face Management: Face-saving, threatening, or enhancing acts
3. HARD NEUTRALITY FILTER (CRITICAL):
   - Force Neutral if: Pure fact relay, agreement, single-word responses, transactional questions
   - Override ONLY if explicit emotional language present
4. Affiliation: Aligning/misaligning stance, solidarity markers
5. Conflict vs Vulnerability distinction

⚠️ GATES:
- Fear: External/time-bound threat or defensive vulnerability
- Sadness: Past loss/resignation or reflective vulnerability
- Default inward distress to Sadness if no clear threat exists

Output tags: [SOCIAL INTENT: HARD NEUTRAL], [SOCIAL INTENT: CONFLICT/AGGRESSION], 
[SOCIAL INTENT: VULNERABILITY/DISTRESS], [SOCIAL INTENT: PLAYFUL/BANTER], [SOCIAL INTENT: MUNDANE/NEUTRAL]

---

## AGENT 7: CHIEF JUSTICE & EMOTION ARBITER (COUNCIL AGGREGATOR)
Role: Synthesize all agent reports into FINAL emotion classification using Chain-of-Thought reasoning.

DECISION GUIDELINES:
1. Calibration: Use "Baseline Arousal" and "Conflict Style" from speaker profiles
2. Emotional Inertia: If [SHIFT: FALSE] and previous confidence > 0.70, maintain previous emotion
3. Valence Filter:
   - Positive → Joy/Surprise
   - Negative → Anger/Sadness/Fear/Disgust
   - Neutral → Neutral/Surprise
4. Temporal Shift: If [SHIFT: TRUE], focus on new trigger
5. Fear Gate: Only if Sentiment is Negative + [SHIFT: TRUE] + Threat language/distress
6. Sadness Anchor: Check for Past-Tense/Resignation ("It's over", "I gave up")
7. Neutrality/Joy Gate: Low-arousal politeness = Neutral. Joy requires achievement/celebration

---

## FINAL OUTPUT FORMAT (STRICT JSON):

For EVERY utterance, provide:
{
  "utterance_id": "exact_id_from_input",
  "predicted_emotion": "single_label_only",
  "confidence": 0.0-1.0,
  "reasoning": "2-sentence Chain-of-Thought explaining choice over alternatives"
}

CRITICAL RULES:
- Return predictions as JSON array under "predictions" key
- ALWAYS include ALL utterance_ids from input
- ONLY output the JSON, no other text
- One emotion per utterance (no multi-label)
- Confidence reflects certainty in final choice

<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

print("✅ Unified System Prompt Loaded")
print(f"  Prompt length: {len(UNIFIED_SYSTEM_PROMPT)} characters")
print(f"  Integrated agents: 7 (Character, Context, Shift, Linguistic, Relational, Social, Aggregator)")

✅ Unified System Prompt Loaded
  Prompt length: 5732 characters
  Integrated agents: 7 (Character, Context, Shift, Linguistic, Relational, Social, Aggregator)


## Section 4: Create Scene Batches with Enriched Context

Build scene batches from dialogue data and enrich with speaker biographical information, scene context, and dialogue history.

In [4]:
def build_scene_batches_with_enriched_context(df: pd.DataFrame, bio_cards: Dict) -> list:
    """
    Build scene batches with enriched context.
    Each scene includes speaker profiles, dialogue, and scene structure.
    """
    scenes = []
    
    for dialogue_id, group in df.groupby("Dialogue_ID", sort=False):
        scene_rows = []
        scene_lines = []
        speakers_in_scene = set()
        
        for _, row in group.iterrows():
            utterance_id = str(row.get("Utterance_ID", "NA"))
            speaker = str(row.get("Speaker", "Unknown"))
            utterance = str(row.get("Utterance", ""))
            gt_emotion = str(row.get("Emotion", "")).strip().lower()
            gt_sentiment = str(row.get("Sentiment", "")).strip().lower()
            
            speakers_in_scene.add(speaker)
            
            scene_rows.append({
                "Dialogue_ID": dialogue_id,
                "Utterance_ID": utterance_id,
                "Recognition_ID": f"{dialogue_id}_{utterance_id}",
                "Speaker": speaker,
                "Utterance": utterance,
                "ground_truth_emotion": gt_emotion,
                "ground_truth_sentiment": gt_sentiment,
            })
            scene_lines.append(f"{utterance_id} | {speaker}: {utterance}")
        
        # Format enriched context
        bio_context = format_bio_cards_context(bio_cards, list(speakers_in_scene))
        scene_dialogue = "\n".join(scene_lines)
        
        # Build user content that provides the dialogue and context to analyze
        user_content = (
            f"Dialogue ID: {dialogue_id}\n"
            f"{bio_context}\n"
            f"### SCENE DIALOGUE\n"
            f"Analyze and predict emotion for each utterance:\n\n"
            f"{scene_dialogue}\n\n"
            f"For EACH utterance_id above, provide your 7-agent integrated analysis and final emotion prediction."
        )
        
        scenes.append({
            "dialogue_id": dialogue_id,
            "rows": scene_rows,
            "user_content": user_content,
            "speakers": list(speakers_in_scene)
        })
    
    return scenes

print("Building scene batches with enriched context...")
scenes = build_scene_batches_with_enriched_context(df, bio_cards)
print(f"\n✅ Scene Batches Ready")
print(f"  Total scenes: {len(scenes)}")
print(f"  Total utterances: {sum(len(s['rows']) for s in scenes)}")
print(f"  Avg utterances per scene: {sum(len(s['rows']) for s in scenes) / len(scenes):.1f}")

Building scene batches with enriched context...

✅ Scene Batches Ready
  Total scenes: 280
  Total utterances: 2610
  Avg utterances per scene: 9.3


## Section 5: Run Inference with Prompt Engineering

Execute Llama 3.1 inference using the unified prompt on scene batches, parse predictions, and save results incrementally.

In [5]:
def call_llama31(system_prompt: str, user_content: str) -> str:
    """Call Llama 3.1 with unified prompt."""
    full_prompt = f"{system_prompt}\n\nUser Input:\n{user_content}"
    response = llama31_model.generate_content(full_prompt)
    return response.text

def parse_llama_predictions(response_text: str) -> Dict[str, Dict]:
    """Extract emotion predictions from Llama 3.1 JSON response."""
    if not response_text:
        return {}
    
    try:
        # Extract JSON from response
        match = re.search(r"\{[\s\S]*\}", response_text)
        if match:
            parsed = json.loads(match.group(0))
        else:
            return {}
    except Exception:
        return {}
    
    if parsed is None:
        return {}
    
    # Extract predictions array
    predictions = []
    if isinstance(parsed, dict):
        maybe_list = parsed.get("predictions", [])
        if isinstance(maybe_list, list):
            predictions = maybe_list
    elif isinstance(parsed, list):
        predictions = parsed
    
    # Map to utterance_id
    out = {}
    for item in predictions:
        if not isinstance(item, dict):
            continue
        utt_id = item.get("utterance_id")
        if utt_id is None:
            continue
        emotion = item.get("predicted_emotion")
        reasoning = item.get("reasoning", "")
        confidence = item.get("confidence", 0.5)
        
        out[str(utt_id)] = {
            "predicted_emotion": emotion.lower().strip() if isinstance(emotion, str) else None,
            "reasoning": str(reasoning) if reasoning is not None else "",
            "confidence": float(confidence) if isinstance(confidence, (int, float)) else 0.5
        }
    return out

def run_inference(scenes: list, limit: Optional[int] = None) -> tuple:
    """
    Run Llama 3.1 unified prompt inference on scene batches.
    Saves results incrementally for safety.
    """
    if limit is not None:
        scenes = scenes[:limit]
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = os.path.join(OUTPUT_DIR, f"llama31_unified_predictions_{timestamp}.csv")
    
    records = []
    total_scenes = len(scenes)
    total_utterances = sum(len(s["rows"]) for s in scenes)
    
    print(f"\n📊 Starting Inference")
    print(f"  Scenes to process: {total_scenes}")
    print(f"  Total utterances: {total_utterances}")
    print(f"  Output: {output_path}\n")
    
    for scene_idx, scene in enumerate(tqdm(scenes, desc="Inference", ncols=80), start=1):
        dialogue_id = scene["dialogue_id"]
        max_retries = 2
        
        for attempt in range(max_retries):
            try:
                # Call Llama 3.1 with unified prompt
                model_output = call_llama31(
                    system_prompt=UNIFIED_SYSTEM_PROMPT,
                    user_content=scene["user_content"]
                )
                
                # Parse predictions
                pred_map = parse_llama_predictions(model_output)
                
                # Build records
                for row in scene["rows"]:
                    pred_item = pred_map.get(str(row["Utterance_ID"]), {})
                    records.append({
                        "Dialogue_ID": row["Dialogue_ID"],
                        "Utterance_ID": row["Utterance_ID"],
                        "Recognition_ID": row["Recognition_ID"],
                        "Speaker": row["Speaker"],
                        "Utterance": row["Utterance"],
                        "ground_truth_emotion": row["ground_truth_emotion"],
                        "ground_truth_sentiment": row["ground_truth_sentiment"],
                        "predicted_emotion": pred_item.get("predicted_emotion"),
                        "reasoning": pred_item.get("reasoning", ""),
                        "confidence": pred_item.get("confidence", 0.5),
                        "model_output": model_output
                    })
                
                # Save incrementally
                results_df = pd.DataFrame(records)
                results_df.to_csv(output_path, index=False)
                break
                
            except Exception as e:
                error_msg = str(e)[:100]
                if attempt < max_retries - 1:
                    print(f"  ⚠️  Scene {dialogue_id} - Attempt {attempt + 1} failed: {error_msg}, retrying...")
                else:
                    print(f"  ❌ Scene {dialogue_id} - Skipped after {max_retries} attempts: {error_msg}")
                continue
    
    # Final save
    results_df = pd.DataFrame(records)
    results_df.to_csv(output_path, index=False)
    
    filled_count = int(results_df["predicted_emotion"].notna().sum())
    print(f"\n✅ Inference Complete")
    print(f"  Predictions saved: {output_path}")
    print(f"  Total records: {len(records)}")
    print(f"  With predictions: {filled_count}")
    
    return output_path, results_df

print("✅ Inference functions ready")

✅ Inference functions ready


In [6]:
# Execute inference on all scenes
# Adjust 'limit' to test on fewer scenes first
limit = None  # Set to e.g., 5 for quick test, None for full run

output_csv, predictions_df = run_inference(scenes, limit=limit)

print(f"\n{'='*80}")
print(f"Results saved to: {output_csv}")
print(f"Total predictions: {len(predictions_df)}")
print(f"\nFirst 5 predictions:")
print(predictions_df[["Dialogue_ID", "Utterance_ID", "Speaker", "Utterance", 
                       "ground_truth_emotion", "predicted_emotion", "confidence"]].head())


📊 Starting Inference
  Scenes to process: 280
  Total utterances: 2610
  Output: ../logs\llama31_unified\llama31_unified_predictions_20260504_221957.csv



Inference: 100%|████████████████████████████| 280/280 [4:54:21<00:00, 63.08s/it]



✅ Inference Complete
  Predictions saved: ../logs\llama31_unified\llama31_unified_predictions_20260504_221957.csv
  Total records: 2610
  With predictions: 2487

Results saved to: ../logs\llama31_unified\llama31_unified_predictions_20260504_221957.csv
Total predictions: 2610

First 5 predictions:
   Dialogue_ID Utterance_ID Speaker  \
0            0            0    Mark   
1            0            1  Rachel   
2            0            2  Rachel   
3            1            0    Joey   
4            1            1    Joey   

                                           Utterance ground_truth_emotion  \
0  Why do all youre coffee mugs have numbers on ...             surprise   
1  Oh. Thats so Monica can keep track. That way ...                anger   
2                                       Y'know what?              neutral   
3                     Come on, Lydia, you can do it.              neutral   
4                                              Push!                  joy   

  p

## Section 6: Merge and Evaluate Results

Merge timestamped prediction files into a master CSV, track progress, and continue on remaining scenes if needed.

In [7]:
import glob

# Merge all timestamped predictions into master file
print("🔄 Merging prediction files...")
pred_files = glob.glob(os.path.join(OUTPUT_DIR, "llama31_unified_predictions_*.csv"))
pred_files = [f for f in pred_files if "master" not in f]

if pred_files:
    print(f"Found {len(pred_files)} prediction files:")
    
    all_dfs = []
    total_rows = 0
    for pred_file in sorted(pred_files):
        df_temp = pd.read_csv(pred_file)
        all_dfs.append(df_temp)
        total_rows += len(df_temp)
        print(f"  - {os.path.basename(pred_file)}: {len(df_temp)} rows")
    
    # Merge and deduplicate
    merged_df = pd.concat(all_dfs, ignore_index=True)
    merged_df = merged_df.drop_duplicates(subset=['Recognition_ID'], keep='first')
    
    print(f"\n✅ Merged {total_rows} rows into {len(merged_df)} unique predictions")
    
    # Save master file
    master_path = os.path.join(OUTPUT_DIR, "llama31_unified_predictions_master.csv")
    merged_df.to_csv(master_path, index=False)
    print(f"✅ Master file saved: {os.path.basename(master_path)}")
    
    results_df = merged_df
else:
    print("❌ No prediction files found!")
    results_df = pd.DataFrame()

# Check for remaining scenes to process
if len(results_df) > 0:
    processed_dialogue_ids = set(results_df['Dialogue_ID'].unique())
    remaining_scenes = [s for s in scenes if s['dialogue_id'] not in processed_dialogue_ids]
    
    print(f"\n📊 Progress Status")
    print(f"  Processed: {len(processed_dialogue_ids)} dialogues, {len(results_df)} utterances")
    print(f"  Remaining: {len(remaining_scenes)} dialogues, {sum(len(s['rows']) for s in remaining_scenes)} utterances")
    
    if remaining_scenes:
        print(f"\n⚡ Continue inference on remaining scenes? (Uncomment and run next cell)")

🔄 Merging prediction files...
Found 1 prediction files:
  - llama31_unified_predictions_20260504_221957.csv: 2610 rows

✅ Merged 2610 rows into 2610 unique predictions
✅ Master file saved: llama31_unified_predictions_master.csv

📊 Progress Status
  Processed: 280 dialogues, 2610 utterances
  Remaining: 0 dialogues, 0 utterances


In [8]:
# Optional: Continue inference on remaining scenes
# Uncomment to run on remaining scenes

# if remaining_scenes:
#     print(f"\n🚀 Processing remaining {len(remaining_scenes)} scenes...")
#     remaining_output_csv, remaining_results_df = run_inference(remaining_scenes, limit=None)
#     
#     # Combine with existing results
#     combined_df = pd.concat([results_df, remaining_results_df], ignore_index=True)
#     master_path = os.path.join(OUTPUT_DIR, "llama31_unified_predictions_master.csv")
#     combined_df.to_csv(master_path, index=False)
#     
#     print(f"\n✅ Updated master file")
#     print(f"  Total predictions: {len(combined_df)}")
#     
#     results_df = combined_df
# else:
#     print("✅ All scenes already processed!")

print("Ready to proceed to evaluation...")

Ready to proceed to evaluation...


## Section 7: Generate Classification Report and Metrics

Load master predictions, compute comprehensive metrics, and display performance analysis.

In [9]:
# Load and evaluate master predictions
master_file = os.path.join(OUTPUT_DIR, "llama31_unified_predictions_master.csv")

if not os.path.exists(master_file):
    print(f"❌ Master file not found: {master_file}")
    print("Please run inference cells first")
else:
    pred_df = pd.read_csv(master_file)
    print(f"✅ Loaded: {len(pred_df)} predictions\n")
    
    # Filter valid predictions
    valid_df = pred_df.dropna(subset=["ground_truth_emotion", "predicted_emotion"]).copy()
    print(f"Valid predictions: {len(valid_df)} / {len(pred_df)}")
    
    if len(valid_df) > 0:
        # Get emotion labels
        valid_emotions = {"anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"}
        all_emotions = sorted(set(
            list(valid_df["ground_truth_emotion"].unique()) + 
            list(valid_df["predicted_emotion"].unique())
        ))
        
        # Normalize emotions
        valid_df["ground_truth_emotion"] = valid_df["ground_truth_emotion"].str.lower().str.strip()
        valid_df["predicted_emotion"] = valid_df["predicted_emotion"].str.lower().str.strip()
        
        print(f"\n{'='*80}")
        print("LLAMA 3.1 UNIFIED PROMPT - CLASSIFICATION REPORT")
        print(f"{'='*80}\n")
        print(f"Emotion classes: {all_emotions}\n")
        
        # Classification report
        report = classification_report(
            valid_df["ground_truth_emotion"],
            valid_df["predicted_emotion"],
            labels=all_emotions,
            digits=4,
            zero_division=0
        )
        print(report)
        
        # Summary metrics
        wf1 = f1_score(valid_df["ground_truth_emotion"], valid_df["predicted_emotion"], 
                       average="weighted", zero_division=0)
        mf1 = f1_score(valid_df["ground_truth_emotion"], valid_df["predicted_emotion"], 
                       average="macro", zero_division=0)
        accuracy = (valid_df["ground_truth_emotion"] == valid_df["predicted_emotion"]).sum() / len(valid_df)
        
        print(f"{'='*80}")
        print("SUMMARY METRICS")
        print(f"{'='*80}")
        print(f"Overall Accuracy:    {accuracy:.4f}")
        print(f"Weighted F1-Score:   {wf1:.4f}")
        print(f"Macro F1-Score:      {mf1:.4f}")
        print(f"Total Valid Predictions: {len(valid_df)}")
        print(f"{'='*80}\n")
        
        # Confusion matrix
        cm = confusion_matrix(valid_df["ground_truth_emotion"], 
                             valid_df["predicted_emotion"], 
                             labels=all_emotions)
        
        print("Confusion Matrix (rows=ground_truth, cols=predicted):")
        print(f"Classes: {all_emotions}")
        print(cm)
        
        # Per-class breakdown
        print(f"\n{'='*80}")
        print("PER-CLASS BREAKDOWN")
        print(f"{'='*80}\n")
        
        for emotion in all_emotions:
            count = (valid_df["ground_truth_emotion"] == emotion).sum()
            correct = ((valid_df["ground_truth_emotion"] == emotion) & 
                      (valid_df["predicted_emotion"] == emotion)).sum()
            accuracy_class = correct / count if count > 0 else 0
            print(f"{emotion:12s} - Count: {count:4d}, Correct: {correct:4d}, Acc: {accuracy_class:.4f}")
    else:
        print("❌ No valid predictions found!")

✅ Loaded: 2610 predictions

Valid predictions: 2487 / 2610

LLAMA 3.1 UNIFIED PROMPT - CLASSIFICATION REPORT

Emotion classes: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

              precision    recall  f1-score   support

       anger     0.7403    0.5859    0.6541       326
     disgust     0.7000    0.4179    0.5234        67
        fear     0.2857    0.6087    0.3889        46
         joy     0.5900    0.7083    0.6438       384
     neutral     0.7992    0.8337    0.8161      1203
     sadness     0.4719    0.6396    0.5431       197
    surprise     0.8148    0.3333    0.4731       264

    accuracy                         0.6980      2487
   macro avg     0.6289    0.5896    0.5775      2487
weighted avg     0.7227    0.6980    0.6944      2487

SUMMARY METRICS
Overall Accuracy:    0.6980
Weighted F1-Score:   0.6944
Macro F1-Score:      0.5775
Total Valid Predictions: 2487

Confusion Matrix (rows=ground_truth, cols=predicted):
Classes: ['anger', '

In [10]:
# Detailed analysis of predictions
print(f"\n{'='*80}")
print("DETAILED ANALYSIS")
print(f"{'='*80}\n")

# Show sample correct and incorrect predictions
print("SAMPLE CORRECT PREDICTIONS:")
correct_mask = valid_df["ground_truth_emotion"] == valid_df["predicted_emotion"]
correct_samples = valid_df[correct_mask][["Speaker", "Utterance", "ground_truth_emotion", 
                                          "predicted_emotion", "confidence"]].head(5)
for idx, row in correct_samples.iterrows():
    print(f"\n  Speaker: {row['Speaker']}")
    print(f"  Utterance: {row['Utterance'][:80]}")
    print(f"  Actual: {row['ground_truth_emotion']:12s} | Predicted: {row['predicted_emotion']:12s} | Conf: {row['confidence']:.4f}")

print(f"\n\nSAMPLE INCORRECT PREDICTIONS:")
incorrect_mask = valid_df["ground_truth_emotion"] != valid_df["predicted_emotion"]
incorrect_samples = valid_df[incorrect_mask][["Speaker", "Utterance", "ground_truth_emotion", 
                                              "predicted_emotion", "confidence"]].head(5)
for idx, row in incorrect_samples.iterrows():
    print(f"\n  Speaker: {row['Speaker']}")
    print(f"  Utterance: {row['Utterance'][:80]}")
    print(f"  Actual: {row['ground_truth_emotion']:12s} | Predicted: {row['predicted_emotion']:12s} | Conf: {row['confidence']:.4f}")

print(f"\n{'='*80}")
print(f"Notebook completed successfully!")
print(f"Master predictions: {master_file}")
print(f"{'='*80}")


DETAILED ANALYSIS

SAMPLE CORRECT PREDICTIONS:

  Speaker: Rachel
  Utterance: Y'know what?
  Actual: neutral      | Predicted: neutral      | Conf: 0.9500

  Speaker: Joey
  Utterance: Push!
  Actual: joy          | Predicted: joy          | Conf: 0.9500

  Speaker: Joey
  Utterance: Push 'em out, push 'em out, harder, harder.
  Actual: joy          | Predicted: joy          | Conf: 0.9800

  Speaker: Joey
  Utterance: Push 'em out, push 'em out, way out!
  Actual: joy          | Predicted: joy          | Conf: 0.9900

  Speaker: Joey
  Utterance: Let's get that ball and really move, hey, hey, ho, ho.
  Actual: joy          | Predicted: joy          | Conf: 0.9900


SAMPLE INCORRECT PREDICTIONS:

  Speaker: Mark
  Utterance: Why do all youre coffee mugs have numbers on the bottom?
  Actual: surprise     | Predicted: neutral      | Conf: 0.9500

  Speaker: Rachel
  Utterance: Oh. Thats so Monica can keep track. That way if one on them is missing, she can
  Actual: anger        | Pre

In [11]:
# Additional analysis and experimentation
# Feel free to add custom analysis here